In [ ]:
import time

from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler


from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import (
    PyPDFLoader, 
    UnstructuredHTMLLoader, 
    UnstructuredWordDocumentLoader
)
from langchain.text_splitter import RecursiveCharacterTextSplitter

from loguru import logger
import shutil
import os
import pandas as pd
import hashlib
from datetime import datetime

import re
import chromadb



In [2]:
WATCH_FOLDER = "docs/landing"
CHROMA_DIR = "../chroma_db"
LOGS_FOLDER = "logs"
DONE_FOLDER = "docs/done"
TRACKING_FILE = "processed_files.csv"

# Create done folder if it doesn't exist
os.makedirs(DONE_FOLDER, exist_ok=True)


In [59]:
logger.add(
    f"{LOGS_FOLDER}/docs_loader.log", 
    rotation="1 week", 
    retention="4 weeks", 
    level="INFO"
)


2

In [60]:

def cleanup_chroma():
    """Clean up ChromaDB and tracking files for fresh start"""
    
    logger.info("Starting ChromaDB cleanup process")
    
    # Remove ChromaDB directory
    if os.path.exists(CHROMA_DIR):
        try:
            shutil.rmtree(CHROMA_DIR)
            logger.success(f"Removed ChromaDB directory: {CHROMA_DIR}")
        except Exception as e:
            logger.error(f"Failed to remove ChromaDB directory: {e}")
    else:
        logger.info(f"ChromaDB directory not found: {CHROMA_DIR}")
    
    # Remove tracking CSV
    tracking_file = "processed_files.csv"
    if os.path.exists(tracking_file):
        try:
            os.remove(tracking_file)
            logger.success(f"Removed tracking file: {tracking_file}")
        except Exception as e:
            logger.error(f"Failed to remove tracking file: {e}")
    else:
        logger.info(f"Tracking file not found: {tracking_file}")
    
    logger.success("Cleanup complete! Ready for fresh document ingestion.")



In [61]:
cleanup_chroma()

2025-08-28 19:07:22.541 | INFO     | __main__:cleanup_chroma:4 - Starting ChromaDB cleanup process
2025-08-28 19:07:22.544 | INFO     | __main__:cleanup_chroma:14 - ChromaDB directory not found: ../chroma_db
2025-08-28 19:07:22.545 | INFO     | __main__:cleanup_chroma:25 - Tracking file not found: processed_files.csv
2025-08-28 19:07:22.546 | SUCCESS  | __main__:cleanup_chroma:27 - Cleanup complete! Ready for fresh document ingestion.


In [24]:
# Get campaign ID from filename
def get_campaign_id_from_filename(filename: str) -> str:
    """
    Extract campaign ID from filename using regex.
    Example filename: "campaign_101_summary_report.pdf"
    """
    match = re.search(r'campaign_(\d+)_', filename)
    campaign_id = match.group(1) if match else None

    return campaign_id
    

In [23]:
filename = "campaign_101_summary_report.pdf"
file_full_path = f"docs/done/{filename}"
campaign_id = get_campaign_id_from_filename(file_full_path)
print(campaign_id)

101


In [26]:
# Ingestion function
MAX_CHARS = 2000  # ~500 tokens

def chunk_text(text, max_chars=MAX_CHARS):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks, current = [], ""

    for sent in sentences:
        if len(current) + len(sent) + 1 <= max_chars:
            current += " " + sent if current else sent
        else:
            chunks.append(current.strip())
            current = sent
    if current:
        chunks.append(current.strip())
    return chunks

def ingest_campaign(doc_text: str, campaign_id: str, metadata: dict, collection):
    """
    Ingest one campaign report into ChromaDB
    """
    headers = [
        "Executive Summary",
        "Campaign Overview",
        "Key Metrics",
        "Performance Insights",
        "Recommendations"
    ]
    pattern = r"(" + "|".join(map(re.escape, headers)) + ")"

    parts = re.split(pattern, doc_text)

    for i in range(1, len(parts), 2):
        header = parts[i].strip()
        content = parts[i+1].strip() if i+1 < len(parts) else ""

        subchunks = chunk_text(content)
        for j, sub in enumerate(subchunks):
            doc_id = f"{campaign_id}_{header.lower().replace(' ','_')}_{j}"
            print("doc_id:", doc_id)
            print("header:", header)
            print("sub:", sub)
            print("campaign_id", campaign_id)
            print("section", header)
            print("part", j)

            
            # collection.add(
            #     documents=[f"{header}\n{sub}"],
            #     ids=[doc_id],
            #     metadatas=[{
            #         **metadata,
            #         "campaign_id": campaign_id,
            #         "section": header,
            #         "part": j
            #     }]
            # )


In [30]:
loader=PyPDFLoader(file_full_path)
pages = loader.load()


In [ ]:
full_text = "\n".join(page.page_content for page in pages)

In [32]:
full_text

'Marketing Campaign Summary Report: Campaign 101\nReport Date: 2025-07-26\nCampaign Date: 2024-06-26\nCustomer Segment: Previous Customers\nCampaign Topic: Loyalty Program\nExecutive Summary\nThe Campaign 101 targeted Previous Customers with the objective to promote loyalty program. Out\nof an audience size of 139,374, emails were sent to 125,675 contacts, with a control group of 13,699.\nThe campaign achieved an open rate of 16.20%, click rate of 4.44%, and a conversion rate of 0.91%,\nsignifying strong engagement and effectiveness.\nCampaign Overview\nCampaign Id\n101\nCampaign Date\n2024-06-26\nCampaign Topic\nLoyalty Program\nCustomer Segment\nPrevious Customers\nAudience Size\n139,374\nControl Group Size\n13,699\nKey Metrics\nMetric\nCount\nRate (%)\nEmails Sent\n125,675\n-\nOpens\n20,361\n16.20\nClicks\n5,582\n4.44\nConversions\n1,146\n0.91\nOpen to Click Rate\n-\n27.42\nPerformance Insights\n- The open rate of 16.20% indicates strong interest among the target audience.\n- An ope

In [45]:
import re

headers = [
    "Executive Summary",
    "Campaign Overview",
    "Key Metrics",
    "Performance Insights",
    "Recommendations"
]

pattern = r"(" + "|".join(map(re.escape, headers)) + ")"
parts = re.split(pattern, full_text)

# Rebuild chunks: combine header + content
sections = []
for i in range(1, len(parts), 2):
    header = parts[i].strip()
    content = parts[i+1].strip() if i+1 < len(parts) else ""
    section = f"{header}\n{content}"
    #print("-----")
    #print(section)
    sections.append(section)

In [42]:
def chunk_text(text, max_chars=2000):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks, current = [], ""
    for sent in sentences:
        if len(current) + len(sent) + 1 <= max_chars:
            current += " " + sent if current else sent
        else:
            chunks.append(current.strip())
            current = sent
    if current:
        chunks.append(current.strip())
    return chunks

final_chunks = []
for section in sections:
    subchunks = chunk_text(section)
    final_chunks.extend(subchunks)

In [43]:
len(final_chunks)

5

In [62]:
# Ingestion function
def ingest_campaign(file_full_path: str):
    """
    Ingest one campaign report into ChromaDB
    """

    def chunk_text(text, max_chars=2000):
        sentences = re.split(r'(?<=[.!?])\s+', text)
        chunks, current = [], ""
        for sent in sentences:
            if len(current) + len(sent) + 1 <= max_chars:
                current += " " + sent if current else sent
            else:
                chunks.append(current.strip())
                current = sent
        if current:
            chunks.append(current.strip())
        return chunks

    # ChromaDB
    client = chromadb.Client()
    collection = client.get_or_create_collection(name="campaign_reports")


    # Metadata specific to campaign
    metadata = {}

    # Get campaign ID from filename
    match = re.search(r'campaign_(\d+)_', filename)
    campaign_id = match.group(1) if match else None

    # Headers
    headers = [
        "Executive Summary",
        "Campaign Overview",
        "Key Metrics",
        "Performance Insights",
        "Recommendations"
    ]

    # Read document
    loader=PyPDFLoader(file_full_path)
    pages = loader.load()
    full_text = "\n".join(page.page_content for page in pages)

    pattern = r"(" + "|".join(map(re.escape, headers)) + ")"
    parts = re.split(pattern, full_text)

    # Rebuild chunks: combine header + content
    sections = []
    for i in range(1, len(parts), 2):
        header = parts[i].strip()
        content = parts[i+1].strip() if i+1 < len(parts) else ""
        section = f"{header}\n{content}"

        subchunks = chunk_text(section)
        for j, sub in enumerate(subchunks):
            doc_id = f"{campaign_id}_{header.lower().replace(' ','_')}_{j}"
            print('-----')
            print("doc_id:", doc_id)
            print("header:", header)
            print("sub:", sub)
            print("campaign_id", campaign_id)
            print("section", header)
            print("part", j)

            
            collection.add(
                documents=[f"{header}\n{sub}"],
                ids=[doc_id],
                metadatas=[{
                    **metadata,
                    "campaign_id": campaign_id,
                    "section": header,
                    "part": j
                }]
            )


In [55]:
filename = "campaign_101_summary_report.pdf"
file_full_path = f"docs/done/{filename}"

In [63]:
ingest_campaign(file_full_path)

-----
doc_id: 101_executive_summary_0
header: Executive Summary
sub: Executive Summary
The Campaign 101 targeted Previous Customers with the objective to promote loyalty program. Out
of an audience size of 139,374, emails were sent to 125,675 contacts, with a control group of 13,699. The campaign achieved an open rate of 16.20%, click rate of 4.44%, and a conversion rate of 0.91%,
signifying strong engagement and effectiveness.
campaign_id 101
section Executive Summary
part 0
-----
doc_id: 101_campaign_overview_0
header: Campaign Overview
sub: Campaign Overview
Campaign Id
101
Campaign Date
2024-06-26
Campaign Topic
Loyalty Program
Customer Segment
Previous Customers
Audience Size
139,374
Control Group Size
13,699
campaign_id 101
section Campaign Overview
part 0
-----
doc_id: 101_key_metrics_0
header: Key Metrics
sub: Key Metrics
Metric
Count
Rate (%)
Emails Sent
125,675
-
Opens
20,361
16.20
Clicks
5,582
4.44
Conversions
1,146
0.91
Open to Click Rate
-
27.42
campaign_id 101
section Key

In [64]:
# Call ingestion function
client = chromadb.Client()
collection = client.get_or_create_collection(name="campaign_reports")


In [82]:
results = collection.query(
    query_texts=["What are the insights for campaign 101?"],
    n_results=1
)

In [83]:
results

{'ids': [['101_campaign_overview_0']],
 'embeddings': None,
 'documents': [['Campaign Overview\nCampaign Overview\nCampaign Id\n101\nCampaign Date\n2024-06-26\nCampaign Topic\nLoyalty Program\nCustomer Segment\nPrevious Customers\nAudience Size\n139,374\nControl Group Size\n13,699']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'section': 'Campaign Overview',
    'campaign_id': '101',
    'part': 0}]],
 'distances': [[0.9073624610900879]]}

In [89]:
query = "What are the insights for Campaign 101?"

In [91]:
# Add campaign_id metadata to each chunk
match = re.search(r'campaign\s+(\d+)', query, re.IGNORECASE)
campaign_id = match.group(1) if match else None
print(campaign_id)


101


In [ ]:
# Query example

# Search across all campaigns
results = collection.query(
    query_texts=["What was the best conversion rate achieved?"],
    n_results=3
)

# Search within a specific campaign
results_campaign101 = collection.query(
    query_texts=["Summarize performance"],
    n_results=3,
    where={"campaign_id": "101"}
)

# Search only inside Key Metrics sections for all campaigns
results_metrics = collection.query(
    query_texts=["What was the click-through rate?"],
    n_results=3,
    where={"section": "Key Metrics"}
)


docs/done/campaign_101_summary_report.pdf
101


In [ ]:


# Initialize text splitter for chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # Characters per chunk
    chunk_overlap=200,  # Overlap between chunks
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)


def get_file_hash(file_path):
    """Generate MD5 hash of file content"""
    with open(file_path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()


def get_processed_files():
    """Get list of already processed files and their hashes from CSV using pandas"""
    if os.path.exists(TRACKING_FILE):
        df = pd.read_csv(TRACKING_FILE)
        # Return both filenames and content hashes
        return set(df['filename'].tolist()), set(df['content_hash'].tolist())
    return set(), set()


def add_processed_file(filename, file_path, content_hash):
    """Add file to processed files CSV using pandas"""
    new_row = pd.DataFrame([{
        'filename': filename,
        'original_path': file_path,
        'content_hash': content_hash,
        'processed_date': datetime.now().isoformat()
    }])
    
    if os.path.exists(TRACKING_FILE):
        df = pd.read_csv(TRACKING_FILE)
        df = pd.concat([df, new_row], ignore_index=True)
    else:
        df = new_row
    
    df.to_csv(TRACKING_FILE, index=False)


def get_loader_for_file(file_path):
    """Get appropriate loader based on file extension"""
    file_ext = os.path.splitext(file_path)[1].lower()
    
    if file_ext == '.pdf':
        return PyPDFLoader(file_path)
    elif file_ext in ['.html', '.htm']:
        return UnstructuredHTMLLoader(file_path)
    elif file_ext == '.docx':
        return UnstructuredWordDocumentLoader(file_path)
    else:
        raise ValueError(f"Unsupported file type: {file_ext}")


class DocumentHandler(FileSystemEventHandler):
    def on_created(self, event):
        if event.is_directory:
            return
        
        file_ext = os.path.splitext(event.src_path)[1].lower()
        if file_ext in ['.pdf', '.html', '.htm', '.docx']:
            filename = os.path.basename(event.src_path)
            content_hash = get_file_hash(event.src_path)
            processed_files, processed_hashes = get_processed_files()
            
            # Check both filename and content hash
            if filename in processed_files or content_hash in processed_hashes:
                logger.info(f"Skipping already processed file: {filename}")
                # Move to done folder without reprocessing
                done_path = os.path.join(DONE_FOLDER, filename)
                shutil.move(event.src_path, done_path)
                logger.info(
                    f"Moved {filename} to done folder (already processed)"
                )
            else:
                logger.info(f"New document detected: {event.src_path}")
                self.ingest_document(event.src_path)

    def ingest_document(self, file_path):
        try:
            filename = os.path.basename(file_path)
            content_hash = get_file_hash(file_path)
            
            # Get appropriate loader
            loader = get_loader_for_file(file_path)
            docs = loader.load()
            
            # Split documents into chunks
            text_chunks = text_splitter.split_documents(docs)
            
            embeddings = HuggingFaceEmbeddings(
                model_name="sentence-transformers/all-MiniLM-L6-v2"
            )
            db = Chroma(
                persist_directory=CHROMA_DIR, 
                embedding_function=embeddings
            )
            db.add_documents(text_chunks)
            logger.success(f"Ingested and indexed: {file_path}")
            
            # Move file to done folder
            done_path = os.path.join(DONE_FOLDER, filename)
            shutil.move(file_path, done_path)
            logger.info(f"Moved {filename} to done folder")
            
            # Add to processed files tracking
            add_processed_file(filename, file_path, content_hash)
            logger.info(f"Added {filename} to processed files tracking")
            
        except Exception as e:
            logger.error(f"Failed to ingest {file_path}: {e}")


if __name__ == "__main__":
    logger.info(f"Starting document watcher for folder: {WATCH_FOLDER}")
    event_handler = DocumentHandler()
    observer = Observer()
    observer.schedule(event_handler, WATCH_FOLDER, recursive=False)
    observer.start()
    logger.info(f"Watching folder: {WATCH_FOLDER} for new documents...")
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        observer.stop()
        logger.info("Shutting down document watcher.")
    observer.join()

In [ ]:
# metadata example
import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions

# Initialize Chroma client and embedding function
client = chromadb.Client(Settings())
embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# Create or get a collection with embedding function
collection = client.get_or_create_collection(
    name="example_collection",
    embedding_function=embedding_func,
    metadata={"hnsw:space": "cosine"}  # Example metadata for collection config
)

# List of documents to add
documents = [
    "The latest iPhone model has impressive features.",
    "Exploring the beaches of Bali is a dream journey.",
    "Leonardo da Vinci's Mona Lisa is iconic."
]

# Corresponding metadata for each document
metadatas = [
    {"category": "technology", "author": "Apple"},
    {"category": "travel", "location": "Bali"},
    {"category": "art", "artist": "Leonardo da Vinci"}
]

# Add documents with metadata and unique ids
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=["doc1", "doc2", "doc3"]
)

# Query the collection with metadata filter
query_result = collection.query(
    query_texts=["Tell me about technology products"],
    n_results=2,
    where={"category": "technology"}  # Metadata filter to retrieve only tech docs
)

print("Queried documents:", query_result['documents'])
print("Queried metadatas:", query_result['metadatas'])
